## Suche mit Heuristik: Greedy und A*

[Video](https://youtu.be/xuD5bSa5qz4)

Die Breitensuche geht vom Start aus gleichmäßig in alle Richtungen, sie weiß nicht, wo das Ziel liegt. Oft können wir
aber schätzen, wie weit ein Zustand noch vom Ziel entfernt ist. Eine solche Schätzung heißt **Heuristik** $h$. Sie
liefert die vermuteten **Vorwärtskosten** eines Zustands, also die Anzahl der Züge, die man vermutlich noch bis zum
Ziel braucht.

Im Labyrinth sind zwei Heuristiken üblich, $(x, y)$ ist der Zustand und $(x_0, y_0)$ das Ziel:

- **Manhattan-Distanz:** $|x_0 - x| + |y_0 - y|$, die Anzahl der Schritte, wenn es keine Wände gäbe.
- **euklidische Distanz:** $\sqrt{(x_0 - x)^2 + (y_0 - y)^2}$, die Luftlinie.

In diesem Notebook verwenden wir die Manhattan-Distanz. Mit der Heuristik bauen wir zwei Suchverfahren:
**Greedy** und **A***.

#### Das Labyrinth

<img src='astar.png'>

Wie bei der Breitensuche ist ein Zustand eine Position $(x, y)$, $x$ ist die Zeile und $y$ die Spalte. Die Funktion
`lade` liest ein Labyrinth aus einer Datei und bestimmt Start und Ziel.

In [1]:
def lade(dateiname):
    global grid, height, width, start, ziel
    f = open(dateiname)
    grid = f.read().splitlines()
    f.close()
    height = len(grid)
    width = len(grid[0])
    for x in range(height):
        for y in range(width):
            if grid[x][y] == 'S':
                start = (x, y)
            elif grid[x][y] == 'Z':
                ziel = (x, y)

lade('maze3.txt')
for zeile in grid:
    print(zeile)
print(f'{start=}, {ziel=}')

##########
#        #
#       Z#
#   S ####
#        #
#        #
##########
start=(3, 4), ziel=(2, 8)


In [2]:
dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]    # oben, unten, links, rechts

def nextstates(state):
    x, y = state
    tmp = []
    for xd, yd in dirs:
        x1 = x + xd
        y1 = y + yd
        if 0 <= x1 < height and 0 <= y1 < width and grid[x1][y1] != '#':
            tmp.append((x1, y1))
    return tmp

def goaltest(state):
    return state == ziel

def h(state):
    '''Manhattan-Distanz zum Ziel: die geschätzten Vorwärtskosten'''
    x, y = state
    x0, y0 = ziel
    return abs(x0 - x) + abs(y0 - y)

def reconstructPath(prev, goalstate):
    state = goalstate
    path = []
    while state is not None:
        path.append(state)
        state = prev[state]
    path.reverse()
    return path

Wir schreiben an jedes freie Feld seinen Heuristik-Wert. Direkt neben dem Ziel steht eine 1. Unter der Mauer steht
eine 2, obwohl man von dort einen Umweg gehen muss. Die Heuristik kennt die Wände nicht.

In [3]:
for x in range(height):
    zeile = ''
    for y in range(width):
        zeile += f'{grid[x][y]:>3}' if grid[x][y] in '#SZ' else f'{h((x, y)):>3}'
    print(zeile)

  #  #  #  #  #  #  #  #  #  #
  #  8  7  6  5  4  3  2  1  #
  #  7  6  5  4  3  2  1  Z  #
  #  8  7  6  S  4  #  #  #  #
  #  9  8  7  6  5  4  3  2  #
  # 10  9  8  7  6  5  4  3  #
  #  #  #  #  #  #  #  #  #  #


#### Der Heap

Bei der Breitensuche nehmen wir immer den Zustand aus der frontier, der am längsten wartet. Jetzt wollen wir den mit
dem **kleinsten Wert** nehmen. Dafür gibt es den **Heap** (Priority Queue), eine Datenstruktur, aus der man das kleinste
Element schnell in $O(\log n)$ entnehmen kann. In einer Liste müsste man dafür jedes Mal alle Elemente ansehen.

In [4]:
import random
random.seed(1)
a = []
for i in range(20):
    a.append(random.randint(0, 100))
min(a)

3

In [5]:
from heapq import heappop, heappush
random.seed(1)
b = []
for i in range(20):
    heappush(b, random.randint(0, 100))
heappop(b)

3

In die frontier legen wir Tupel aus Bewertung und Zustand. Tupel werden zuerst nach dem ersten Element verglichen.
Bei gleicher Bewertung entscheidet der Zustand.

In [6]:
b = []
heappush(b, (7, (4, 4)))
heappush(b, (7, (3, 3)))
heappush(b, (5, (3, 5)))
heappush(b, (5, (2, 4)))
print(heappop(b))
print(heappop(b))

(5, (2, 4))
(5, (3, 5))


Zum Vergleich messen wir, wie viele Zustände ein Verfahren findet, also die Länge von `prev`. Die Funktion `zeige`
markiert den gefundenen Weg mit `o` und die übrigen gefundenen Zustände mit `.`.

In [7]:
def zeige(prev, goal):
    path = reconstructPath(prev, goal)
    bild = [list(zeile) for zeile in grid]
    for x, y in prev:
        if bild[x][y] == ' ':
            bild[x][y] = '.'
    for x, y in path[1:-1]:
        bild[x][y] = 'o'
    for zeile in bild:
        print(''.join(zeile))
    print('Weglänge:', len(path) - 1, ' gefundene Zustände:', len(prev))

Zuerst die Breitensuche aus dem Notebook `search_bfs`:

In [8]:
from collections import deque

def bfs(startstate):
    frontier = deque([startstate])
    prev = {startstate: None}
    while frontier:
        state = frontier.popleft()
        if goaltest(state):
            return prev, state
        for v in nextstates(state):
            if v not in prev:
                frontier.append(v)
                prev[v] = state
    return None, None

prev, goal = bfs(start)
zeige(prev, goal)

##########
#........#
#...ooooZ#
#...S.####
#........#
#....... #
##########
Weglänge: 5  gefundene Zustände: 36


#### Greedy

Greedy (gierig) nimmt immer den Zustand aus der frontier, der laut Heuristik am nächsten am Ziel ist. Die frontier ist
ein Heap mit Tupeln `(h(v), v)`. Sonst ändert sich nichts gegenüber der Breitensuche.

In [9]:
def greedy(startstate):
    frontier = [(h(startstate), startstate)]
    prev = {startstate: None}
    while frontier:
        _, state = heappop(frontier)
        if goaltest(state):
            return prev, state
        for v in nextstates(state):
            if v not in prev:
                heappush(frontier, (h(v), v))
                prev[v] = state
    return None, None

prev, goal = greedy(start)
zeige(prev, goal)

##########
#   .... #
#  .ooooZ#
#  .S.####
#   .    #
#        #
##########
Weglänge: 5  gefundene Zustände: 14


Greedy findet hier denselben Weg wie die Breitensuche, braucht dafür aber viel weniger Zustände. Das geht nicht immer
gut. Im folgenden Labyrinth lockt die Heuristik Greedy nach rechts auf einen langen Umweg:

In [10]:
lade('maze4.txt')
prev, goal = bfs(start)
zeige(prev, goal)
print()
prev, goal = greedy(start)
zeige(prev, goal)

############
#oooooooooo#
#o########o#
#o#.......o#
#o#.######o#
#o#......#o#
#o######.#o#
#S.......#Z#
############
Weglänge: 21  gefundene Zustände: 44

############
#          #
# ########.#
# #oooooooo#
# #o######o#
#.#oooooo#o#
#.######o#o#
#Sooooooo#Z#
############
Weglänge: 27  gefundene Zustände: 31


Greedy findet einen Weg mit 27 Zügen, der kürzeste hat 21. Greedy achtet nur darauf, wie weit es noch bis zum Ziel
ist, aber nicht darauf, wie lang der Weg bis hierher schon war.

#### A*

A* (gesprochen A-Stern) berücksichtigt beides:

- $g(v)$: die **Rückwärtskosten**, die Anzahl der Züge vom Start bis $v$ auf dem bisher besten bekannten Weg,
- $h(v)$: die geschätzten **Vorwärtskosten** laut Heuristik.

Die Summe $f(v) = g(v) + h(v)$ ist der **A*-Wert**, die geschätzte Länge des ganzen Wegs über $v$. Aus der frontier
holen wir immer den Zustand mit dem kleinsten A*-Wert.

Im Bild oben steht an jedem Zustand blau die Heuristik $h$ und rot der A*-Wert. Vom Start $(3, 4)$ aus haben
$(2, 4)$ und $(3, 5)$ den A*-Wert $1 + 4 = 5$, $(4, 4)$ und $(3, 3)$ den Wert $1 + 6 = 7$.

In [11]:
def astar(startstate):
    frontier = [(h(startstate), startstate)]
    prev = {startstate: None}
    g = {startstate: 0}                  # Rückwärtskosten: die Anzahl der Züge
    while frontier:
        _, state = heappop(frontier)
        if goaltest(state):
            return prev, state
        for v in nextstates(state):
            gg = g[state] + 1
            if v not in prev or gg < g[v]:
                g[v] = gg
                heappush(frontier, (g[v] + h(v), v))
                prev[v] = state
    return None, None

lade('maze3.txt')
prev, goal = astar(start)
zeige(prev, goal)

##########
#   .... #
#  .ooooZ#
#  .S.####
#   .    #
#        #
##########
Weglänge: 5  gefundene Zustände: 14


In [12]:
lade('maze4.txt')
prev, goal = astar(start)
zeige(prev, goal)

############
#oooooooooo#
#o########o#
#o#      .o#
#o# ######o#
#o#......#o#
#o######.#o#
#S.......#Z#
############
Weglänge: 21  gefundene Zustände: 37


In der Falle findet A* wie die Breitensuche den kürzesten Weg mit 21 Zügen, findet dabei aber weniger Zustände.

#### Erläuterung zum Code

Neu gegenüber der Breitensuche ist das Dictionary `g` mit den Rückwärtskosten. Für jeden Folgezustand $v$ berechnen wir
die Kosten `gg`, wenn wir über `state` nach $v$ gehen.

Falls $v$ nicht in `prev` ist, wurde $v$ zum ersten Mal gefunden:

<img src='astar1.png'>

Falls $v$ schon in `prev` ist, kann es sein, dass der neue Weg über `state` billiger ist als der bisher bekannte. Dann
werden `g[v]` und `prev[v]` geändert, und $v$ kommt mit dem kleineren A*-Wert noch einmal in die frontier:

<img src='astar2.png'>

Im Labyrinth kostet jeder Schritt 1. Wenn Schritte verschiedene Kosten haben, ersetzt man `g[state] + 1` durch
`g[state] + cost(state, v)`. Das Bild zeigt so einen Fall.

Der alte Eintrag von $v$ bleibt dabei im Heap liegen. Er wird später herausgeholt und noch einmal untersucht. Das
kostet etwas Zeit, ändert am Ergebnis aber nichts.

#### Zulässige Heuristiken

Eine Heuristik heißt **zulässig**, wenn sie die tatsächlichen Kosten bis zum Ziel nie überschätzt. Mit einer zulässigen
Heuristik findet A* immer einen kürzesten Weg. Die Manhattan-Distanz ist im Labyrinth zulässig, denn jede Wand kann den
Weg nur verlängern. Auch die euklidische Distanz ist zulässig, sie ist nie größer als die Manhattan-Distanz.

<img src='astar3.png' width='400'>

Das Bild zeigt den Unterschied: Die Breitensuche untersucht Ebene für Ebene alles in der Nähe des Starts, A* geht
gezielt in Richtung Ziel.

Überschätzt die Heuristik, geht die Garantie verloren. Wir nehmen in der Falle das Dreifache der Manhattan-Distanz:

In [13]:
def h(state):
    x, y = state
    x0, y0 = ziel
    return 3 * (abs(x0 - x) + abs(y0 - y))

lade('maze4.txt')
prev, goal = astar(start)
zeige(prev, goal)

############
#.         #
#.########.#
#.#oooooooo#
#.#o######o#
#.#oooooo#o#
#.######o#o#
#Sooooooo#Z#
############
Weglänge: 27  gefundene Zustände: 35


Jetzt findet A* wie Greedy nur einen Weg mit 27 Zügen. Je stärker die Heuristik gewichtet wird, desto mehr verhält
sich A* wie Greedy.

In [14]:
def h(state):
    '''wieder die Manhattan-Distanz'''
    x, y = state
    x0, y0 = ziel
    return abs(x0 - x) + abs(y0 - y)

#### Das A*-Muster

Wie bei der Breitensuche bleiben `astar`, `reconstructPath` und `getMoves` für jedes Problem gleich. Zu schreiben sind

- die Modellierung des Zustands,
- `nextstates` und `goaltest`,
- die Heuristik `h`, die zulässig sein sollte,
- falls gewünscht `getMove`.

```python
def h(state):
    '''returns: die geschätzten Vorwärtskosten von state bis zum Ziel'''

# Aufruf:
prev, goal = astar(startstate)
path = reconstructPath(prev, goal)
print(getMoves(path))
```

#### A* beim 8-Puzzle

Die Breitensuche hat für das 8-Puzzle aus dem Notebook `search_bfs` 177809 Zustände gefunden. Wir probieren zwei
Heuristiken aus:

- **falsche Plättchen:** die Anzahl der Plättchen, die nicht an ihrem Zielplatz stehen,
- **Manhattan-Summe:** für jedes Plättchen die Anzahl der Zeilen und Spalten bis zu seinem Zielplatz, aufaddiert.

Beide sind zulässig: Jedes falsche Plättchen muss mindestens einmal bewegt werden, und ein Zug bewegt ein Plättchen
um genau ein Feld. Die Leerstelle zählt nicht mit.

In [15]:
def goaltest(state):
    return state == (0, 1, 2, 3, 4, 5, 6, 7, 8)

def nextstates(state):
    tmp = []
    i = state.index(0)
    for d in [-3, 3, -1, 1]:
        j = i + d
        if 0 <= j < 9 and (i % 3 == j % 3 or i // 3 == j // 3):
            a = list(state)
            a[i], a[j] = a[j], a[i]
            tmp.append(tuple(a))
    return tmp

def falsche_plaettchen(state):
    anzahl = 0
    for i in range(9):
        if state[i] != 0 and state[i] != i:
            anzahl += 1
    return anzahl

def manhattan_summe(state):
    summe = 0
    for i in range(9):
        zahl = state[i]
        if zahl != 0:
            summe += abs(i // 3 - zahl // 3) + abs(i % 3 - zahl % 3)
    return summe

startstate = (7, 2, 4, 5, 0, 6, 8, 3, 1)
print(falsche_plaettchen(startstate), manhattan_summe(startstate))

8 18


Im Zielzustand steht die Zahl $k$ an Index $k$, also in Zeile $k // 3$ und Spalte $k \% 3$. Das nutzt
`manhattan_summe`.

In [16]:
for heuristik in [falsche_plaettchen, manhattan_summe]:
    h = heuristik
    prev, goal = astar(startstate)
    path = reconstructPath(prev, goal)
    print(f'A* mit {heuristik.__name__}: {len(path) - 1} Züge, {len(prev)} gefundene Zustände')

prev, goal = greedy(startstate)
path = reconstructPath(prev, goal)
print(f'Greedy mit manhattan_summe: {len(path) - 1} Züge, {len(prev)} gefundene Zustände')

A* mit falsche_plaettchen: 26 Züge, 46966 gefundene Zustände
A* mit manhattan_summe: 26 Züge, 3081 gefundene Zustände
Greedy mit manhattan_summe: 28 Züge, 124 gefundene Zustände


Beide A*-Varianten finden die kürzeste Lösung mit 26 Zügen. Mit der Manhattan-Summe sind es 3081 statt 177809
gefundene Zustände. Je genauer eine zulässige Heuristik die wahren Kosten schätzt, desto weniger muss A* untersuchen.
Greedy ist noch schneller, findet aber eine Lösung mit 28 statt 26 Zügen.

#### Zum Weiterüben

Codingame:

- [A-star exercise](https://www.codingame.com/training/medium/a-star-exercise)
- [11 Puzzle](https://www.codingame.com/training/hard/11-puzzle)